# MOD-03 · NB-02 — Relative Risk, Odds Ratios & Confidence Intervals
### Health Analytics with Python · Module 03: Statistical Inference
---
**Learning objectives**
- Calculate RR, OR, ARR, NNT from 2×2 contingency tables
- Construct bootstrap and asymptotic 95% CIs for all four measures
- Build and interpret forest plots for multiple exposure-outcome pairs
- Understand when OR approximates RR (and when it does not)
- Apply the Mantel-Haenszel method for stratified analysis

**Estimated time:** 2 hours | **Level:** Intermediate | **Libraries:** `pandas`, `numpy`, `scipy`, `matplotlib`


## 1. Setup & Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import chi2_contingency, norm
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi':120,'figure.facecolor':'white',
                     'axes.spines.top':False,'axes.spines.right':False})

np.random.seed(42); N=1200
def logistic(x): return 1/(1+np.exp(-x))
base = np.random.normal(size=(N,3))
np.random.seed(99)
diabetes      = np.random.binomial(1,logistic(0.6*base[:,0]-0.5),N)
hypertension  = np.random.binomial(1,logistic(0.7*base[:,0]-0.2),N)
heart_failure = np.random.binomial(1,logistic(0.4*base[:,1]+0.5*hypertension-1.5),N)
obesity       = np.random.binomial(1,logistic(0.5*base[:,0]-0.8),N)
ckd           = np.random.binomial(1,logistic(0.5*base[:,0]+0.6*diabetes-1.2),N)
copd          = np.random.binomial(1,logistic(0.5*base[:,2]-1.0),N)
los_days      = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
ages          = np.random.normal(62,16,N).clip(18,95).astype(int)
sexes         = np.random.choice(['M','F'],N,p=[0.48,0.52])
readmitted_30 = np.random.binomial(1,logistic(-2.5+0.4*heart_failure+0.3*diabetes+0.2*(los_days>7)+0.2*ckd),N)
payers        = np.random.choice(['Medicare','Medicaid','Private','Self-pay','Other'],N,p=[0.40,0.22,0.28,0.07,0.03])

df = pd.DataFrame({
    'patient_id':[f'PT-{i:05d}' for i in range(1,N+1)],
    'age':ages,'sex':sexes,'payer':payers,'los_days':los_days,
    'diabetes':diabetes,'hypertension':hypertension,'heart_failure':heart_failure,
    'obesity':obesity,'ckd':ckd,'copd':copd,'readmitted_30':readmitted_30,
})
df['age_group']=pd.cut(df['age'],[17,44,64,74,95],labels=['18-44','45-64','65-74','75+'])
print(f"Dataset: {df.shape}  |  Outcome (readmission) prevalence: {readmitted_30.mean()*100:.1f}%")


## 2. Core Measures from a 2×2 Table

In [ ]:
def measures_2x2(exposed, outcome, exposure_label='Exposed', ci_method='asymptotic'):
    """
    Compute RR, OR, ARR, and NNT with 95% CI from binary exposure and outcome.
    Returns a DataFrame suitable for forest plots.
    """
    a = int(((exposed==1)&(outcome==1)).sum())  # exposed + event
    b = int(((exposed==1)&(outcome==0)).sum())  # exposed + no event
    c = int(((exposed==0)&(outcome==1)).sum())  # unexposed + event
    d = int(((exposed==0)&(outcome==0)).sum())  # unexposed + no event
    n_e, n_u = a+b, c+d
    risk_e, risk_u = a/n_e, c/n_u

    # Point estimates
    ARR = risk_e - risk_u          # absolute risk reduction (negative = harm)
    NNT = 1/abs(ARR) if ARR != 0 else np.inf
    RR  = risk_e/risk_u
    OR  = (a*d)/(b*c) if b*c>0 else np.nan

    # 95% CI (delta method / log)
    z = norm.ppf(0.975)
    se_log_rr = np.sqrt((b/(a*n_e)) + (d/(c*n_u)))
    rr_lo, rr_hi = np.exp(np.log(RR) - z*se_log_rr), np.exp(np.log(RR) + z*se_log_rr)
    se_log_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
    or_lo, or_hi = np.exp(np.log(OR) - z*se_log_or), np.exp(np.log(OR) + z*se_log_or)
    se_arr = np.sqrt(risk_e*(1-risk_e)/n_e + risk_u*(1-risk_u)/n_u)
    arr_lo, arr_hi = ARR - z*se_arr, ARR + z*se_arr

    _, p_val, _, _ = chi2_contingency([[a,b],[c,d]])

    return {
        'Exposure'    : exposure_label,
        'N_exposed'   : n_e,
        'N_unexposed' : n_u,
        'Risk_exposed': round(risk_e*100,1),
        'Risk_unexpos': round(risk_u*100,1),
        'ARR_pct'     : round(ARR*100,2),
        'NNT'         : round(NNT,1),
        'RR'          : round(RR,2),
        'RR_lo'       : round(rr_lo,2),
        'RR_hi'       : round(rr_hi,2),
        'OR'          : round(OR,2),
        'OR_lo'       : round(or_lo,2),
        'OR_hi'       : round(or_hi,2),
        'p_value'     : round(p_val,4),
    }

# Single worked example: diabetes → readmission
result = measures_2x2(df['diabetes'], df['readmitted_30'], 'Diabetes')
print("Worked example — Diabetes → 30-day readmission")
print(f"  Risk (exposed)  : {result['Risk_exposed']}%")
print(f"  Risk (unexposed): {result['Risk_unexpos']}%")
print(f"  ARR             : {result['ARR_pct']}% (absolute risk difference)")
print(f"  NNT             : {result['NNT']}  (patients needed to treat/screen to observe 1 event)")
print(f"  RR              : {result['RR']} (95% CI: {result['RR_lo']}–{result['RR_hi']})")
print(f"  OR              : {result['OR']} (95% CI: {result['OR_lo']}–{result['OR_hi']})")
print(f"  p-value         : {result['p_value']}")


## 3. Bootstrap Confidence Intervals

In [ ]:
def bootstrap_rr_or(exposed, outcome, n_boot=2000, seed=42):
    """Bootstrap 95% CI for RR and OR — distribution-free."""
    rng = np.random.default_rng(seed)
    idx = np.arange(len(exposed))
    rrs, ors = [], []
    for _ in range(n_boot):
        boot_idx = rng.choice(idx, size=len(idx), replace=True)
        e = exposed.iloc[boot_idx].values if hasattr(exposed,'iloc') else exposed[boot_idx]
        o = outcome.iloc[boot_idx].values if hasattr(outcome,'iloc') else outcome[boot_idx]
        a,b,c,d = ((e==1)&(o==1)).sum(),((e==1)&(o==0)).sum(),                   ((e==0)&(o==1)).sum(),((e==0)&(o==0)).sum()
        n_e,n_u = a+b,c+d
        if n_e>0 and n_u>0 and a>0 and c>0 and b>0 and d>0:
            rrs.append((a/n_e)/(c/n_u))
            ors.append((a*d)/(b*c))
    rr_ci = np.percentile(rrs,[2.5,97.5])
    or_ci = np.percentile(ors,[2.5,97.5])
    return rr_ci, or_ci, np.array(rrs), np.array(ors)

rr_ci, or_ci, rrs, ors = bootstrap_rr_or(df['diabetes'], df['readmitted_30'])
asym = measures_2x2(df['diabetes'], df['readmitted_30'], 'Diabetes')

print("Comparison: Asymptotic vs Bootstrap 95% CI for Diabetes → Readmission")
print(f"  RR (asymptotic): {asym['RR']}  ({asym['RR_lo']}–{asym['RR_hi']})")
print(f"  RR (bootstrap) : {np.median(rrs):.2f}  ({rr_ci[0]:.2f}–{rr_ci[1]:.2f})")
print(f"  OR (asymptotic): {asym['OR']}  ({asym['OR_lo']}–{asym['OR_hi']})")
print(f"  OR (bootstrap) : {np.median(ors):.2f}  ({or_ci[0]:.2f}–{or_ci[1]:.2f})")

fig,axes = plt.subplots(1,2,figsize=(12,4))
for ax, vals, point, label in [
        (axes[0], rrs, asym['RR'], 'RR'),
        (axes[1], ors, asym['OR'], 'OR')]:
    ax.hist(vals, bins=50, color='#4878CF', alpha=0.75, edgecolor='white')
    ax.axvline(point, color='#1F4E79', lw=2, label=f'Point estimate ({point})')
    lo,hi = np.percentile(vals,[2.5,97.5])
    ax.axvline(lo, color='red', ls='--', lw=1.2, label=f'2.5th pct ({lo:.2f})')
    ax.axvline(hi, color='red', ls='--', lw=1.2, label=f'97.5th pct ({hi:.2f})')
    ax.set_xlabel(label); ax.set_ylabel('Bootstrap frequency')
    ax.set_title(f'Bootstrap distribution — {label}'); ax.legend(fontsize=8)
plt.tight_layout()
import os; os.makedirs('/tmp/mod03',exist_ok=True)
plt.savefig('/tmp/mod03/bootstrap_ci.png',bbox_inches='tight'); plt.show()


## 4. Forest Plot — Multiple Exposures

In [ ]:
exposures = [
    ('diabetes','Diabetes'),('heart_failure','Heart failure'),
    ('hypertension','Hypertension'),('obesity','Obesity'),
    ('ckd','CKD'),('copd','COPD'),
]
rows = [measures_2x2(df[col], df['readmitted_30'], label) for col,label in exposures]
forest_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1,2,figsize=(15,6),sharey=True)
fig.suptitle('Association between comorbidities and 30-day readmission', fontsize=13)

y_pos = range(len(forest_df)-1,-1,-1)  # reverse for top-to-bottom

for ax, metric, lo_col, hi_col, null_line, xlabel in [
        (axes[0],'RR','RR_lo','RR_hi',1.0,'Risk Ratio (RR)'),
        (axes[1],'OR','OR_lo','OR_hi',1.0,'Odds Ratio (OR)'),
]:
    for i,(_, row) in enumerate(forest_df.iterrows()):
        color = '#D65F5F' if row[metric] > null_line and row['p_value'] < 0.05 else '#4878CF'
        y = list(y_pos)[i]
        ax.plot([row[lo_col],row[hi_col]],[y,y],color=color,lw=2)
        ax.plot(row[metric],y,'o',color=color,ms=8,zorder=5)
        ax.text(max(row[hi_col], null_line)+0.04, y,
                f"{row[metric]:.2f} ({row[lo_col]:.2f}–{row[hi_col]:.2f})",
                va='center',fontsize=8)
    ax.axvline(null_line,color='black',ls='--',lw=1.2,label=f'{metric}=1 (null)')
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(forest_df['Exposure'])
    ax.set_xlabel(xlabel,fontsize=11); ax.legend(fontsize=9)
    ax.set_title(f'{metric} with 95% CI')
    ax.set_xlim(0.5, forest_df[hi_col].max()+0.6)

plt.tight_layout()
plt.savefig('/tmp/mod03/forest_plot.png',bbox_inches='tight'); plt.show()


## 5. When OR ≈ RR (and When It Doesn't)

In [ ]:
print("OR vs RR: the rare-disease assumption")
print("="*55)
print(f"{'Outcome prevalence':>22s}  {'RR':>6s}  {'OR':>6s}  {'OR/RR':>7s}  {'OR ≈ RR?':>9s}")
print("-"*55)

np.random.seed(7)
for prev_target in [0.02, 0.05, 0.10, 0.20, 0.35]:
    # Simulate so that unexposed prevalence = prev_target and exposed = 1.5×
    risk_u = prev_target
    risk_e = min(risk_u * 1.5, 0.99)
    exp    = np.random.binomial(1, 0.5, 5000)
    prob   = np.where(exp==1, risk_e, risk_u)
    out    = np.random.binomial(1, prob, 5000)
    r      = measures_2x2(exp, out, '')
    ratio  = r['OR']/r['RR'] if r['RR']>0 else np.nan
    approx = '✓ Yes' if abs(ratio-1) < 0.10 else '✗ No'
    print(f"  Prevalence={prev_target*100:.0f}%:       {r['RR']:>6.2f}  {r['OR']:>6.2f}  {ratio:>7.2f}  {approx:>9s}")

print()
print("Rule: OR ≈ RR when outcome prevalence < 10%.")
print("When prevalence > 10%, OR OVER-estimates RR — report both or convert.")


## 6. Mantel-Haenszel Stratified Analysis

In [ ]:
# Stratified analysis: control for age confounding in diabetes → readmission
age_strata = ['18-44','45-64','65-74','75+']

# Stratum-specific 2×2 tables
stratum_results = []
for stratum in age_strata:
    sub = df[df['age_group']==stratum]
    a = ((sub.diabetes==1)&(sub.readmitted_30==1)).sum()
    b = ((sub.diabetes==1)&(sub.readmitted_30==0)).sum()
    c = ((sub.diabetes==0)&(sub.readmitted_30==1)).sum()
    d = ((sub.diabetes==0)&(sub.readmitted_30==0)).sum()
    n = a+b+c+d
    or_s = (a*d)/(b*c) if b*c>0 else np.nan
    rr_s = (a/(a+b))/(c/(c+d)) if (c+d)>0 else np.nan
    stratum_results.append({'Stratum':stratum,'a':a,'b':b,'c':c,'d':d,'n':n,
                            'OR_stratum':round(or_s,2),'RR_stratum':round(rr_s,2)})

sr_df = pd.DataFrame(stratum_results)
print("Stratum-specific estimates — Diabetes → Readmission:")
display(sr_df[['Stratum','n','RR_stratum','OR_stratum']])

# Mantel-Haenszel pooled OR
numerator   = sum(r['a']*r['d']/r['n'] for _,r in sr_df.iterrows())
denominator = sum(r['b']*r['c']/r['n'] for _,r in sr_df.iterrows())
OR_MH = numerator/denominator
print(f"
Mantel-Haenszel pooled OR: {OR_MH:.3f}")
crude_result = measures_2x2(df['diabetes'], df['readmitted_30'], 'Diabetes (crude)')
print(f"Crude OR               : {crude_result['OR']}")
print(f"Difference (confounding by age): {OR_MH - crude_result['OR']:+.3f}")
print()
print("If MH pooled OR ≈ crude OR → age is NOT a major confounder here.")
print("If MH pooled OR ≠ crude OR → age IS confounding the association.")


## 7. Knowledge Check
1. You observe OR=3.1 for smoking → lung cancer in a case-control study (case prevalence 45%). Should you report RR? Why or why not?
2. ARR = -2.3% for a treatment vs control. What is the NNT, and how do you interpret it?
3. In the forest plot, CKD has a wide CI. What does this tell you?
4. When would you prefer bootstrap CI over the asymptotic log method?
5. Diabetes OR = 1.8 (crude). After stratifying by age the MH OR = 2.2. What does this imply?


In [ ]:
# Exercise 5 — simulate confounding and test MH vs crude
np.random.seed(77)
n_ex = 2000
age_ex   = np.random.binomial(1,0.5,n_ex)            # older=1
# Older → more diabetes AND more readmission (positive confounding)
diab_ex  = np.random.binomial(1,logistic(-1+1.5*age_ex),n_ex)
read_ex  = np.random.binomial(1,logistic(-2.5+0.4*diab_ex+1.0*age_ex),n_ex)

crude   = measures_2x2(pd.Series(diab_ex),pd.Series(read_ex),'Crude')
# MH by age stratum
for stratum_name, mask in [('Young (age=0)', age_ex==0),('Old (age=1)', age_ex==1)]:
    r = measures_2x2(pd.Series(diab_ex[mask]),pd.Series(read_ex[mask]),stratum_name)
    print(f"{stratum_name:20s}: OR={r['OR']}")
print(f"Crude OR: {crude['OR']}")
print("If stratum-specific ORs are smaller than crude → positive confounding by age.")


---
**Next → NB-03: Survival Analysis (Kaplan-Meier & Log-Rank)**